# Problem 2 - Wine Quality Prediction on AWS SageMaker

In [1]:
%pip install --force-reinstall --no-cache-dir "sagemaker>=2,<3"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 75.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 237.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 304.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 354.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 627.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 112.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 199.0 MB/s  0:00:00
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.9.3-py3-none-any.whl size=144590 sha256=491e9fc7f317364ad6a1f445b71125aa73f7608b021f351a82505bec04561363
  Stored in directory: /tmp/pip-ephem-wheel-cache-l1bdjp6m/wheels/1f/be/48/13754633f1d08d1fbfc60d5e80ae1e5d7329500477685286cd
Successfully built ant

In [2]:
import os
import pandas as pd
import numpy as np
import sagemaker

# Initialize SageMaker

session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = session.default_bucket()
prefix = 'wine-quality'

# Read Datasets

red_df = pd.read_csv('shared/winequality-red.csv', sep=';')
white_df = pd.read_csv('shared/winequality-white.csv', sep=';')


# Combine Datasets
df = pd.concat([red_df, white_df], axis=0, ignore_index=True)

# Reorder columns

df_sm = pd.concat([df['quality'], df.drop(columns=['quality'])], axis=1)

# Save
os.makedirs('data', exist_ok=True)
df_sm.to_csv('data/wine_data.csv', header=False, index=False)

# Uploaded 
train_input = session.upload_data('data', key_prefix=f'{prefix}/data')
print("Uploaded to:", train_input)

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
Uploaded to: s3://amazon-sagemaker-365698175161-us-east-2-cu2ccb8aeza2sp/shared/wine-quality/data


## Part (b) - Linear Regression WITH Conatiner Technology in SageMaker

In [7]:
from sagemaker.sklearn.estimator import SKLearn


#Define Estimator
sklearn_estimator = SKLearn(
    entry_point='train_wine.py',
    source_dir='shared',
    role=role,
    instance_type='ml.m5.large',
    framework_version='1.2-1',
    py_version='py3',
    sagemaker_session=session,
    disable_profiler=True,
    debugger_hook_config=False
)

# Train Model

sklearn_estimator.fit({'train': train_input})

# Deploy

predictor = sklearn_estimator.deploy(initial_instance_count=1, instance_type='ml.m5.large')


print("Deployed Successfully")


sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
2026-08-13 05:54:19 Starting - Starting the training job...
2026-08-13 05:54:51 Downloading - Downloading input data...
2026-08-13 05:55:16 Downloading - Downloading the training image......
2026-08-13 05:56:17 Training - Training image download completed. Training in progress../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-08-13 05:56:20,386 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2026-08-13 05:56:20,391 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-08-13 05:56:20,393 sagemaker-training-toolkit INFO     No Neurons de

In [8]:
predictor.predict([[7.4, 0.70, 0.00, 1.9, 0.076, 11.0, 34.0, 0.9978, 3.51, 0.56, 9.4]])

array([4.98377854])

## Part (a) - Linear Regression WITHOUT Container Technology

In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# Train and val

train_data, val_data = train_test_split(df_sm, test_size=0.2, random_state=42)

X_train = train_data.iloc[:, 1:]
y_train = train_data.iloc[:, 0]
X_val = val_data.iloc[:, 1:]
y_val = val_data.iloc[:, 0]

lr = LinearRegression()
lr.fit(X_train, y_train)
preds = lr.predict(X_val)

print("R Squared:", r2_score(y_val, preds))
print("RMSE:", mean_squared_error(y_val, preds) ** 0.5)

R Squared: 0.2597673129790178
RMSE: 0.7393892357602032
